In [ ]:
# import os
# import pandas as pd

# folder_path = "final_features"
# merged_df = pd.DataFrame()

# for file in os.listdir(folder_path):
#     if file.startswith("final_") and file.endswith(".csv"):
#         symbol = file.replace("final_", "").replace(".csv", "")
#         df = pd.read_csv(os.path.join(folder_path, file))
#         df["symbol"] = symbol
#         merged_df = pd.concat([merged_df, df], ignore_index=True)

# # Optional: save to CSV
# merged_df.to_csv("merged_all_symbols.csv", index=False)
# print("✅ Merged all symbols. Shape:", merged_df.shape)

# Finetuning

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

df = pd.read_csv("merged_all_symbols.csv")

In [ ]:
df.shape

In [ ]:
df.isna().sum()[df.isna().sum() > 0]

In [ ]:
df['date'] = pd.to_datetime(df['date'])
df.sort_values(by=['symbol', 'date'], inplace=True) 

In [ ]:
features = [col for col in df.columns if col not in ['date', 'symbol', 'close']]
target_col = 'close'

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler_dict = {}
for sym in df['symbol'].unique():
    sym_mask = df['symbol'] == sym
    cols_to_scale = features + [target_col]
    
    df.loc[sym_mask, cols_to_scale] = df.loc[sym_mask, cols_to_scale].astype(float)
    scaler = StandardScaler()
    df.loc[sym_mask, cols_to_scale] = scaler.fit_transform(
        df.loc[sym_mask, cols_to_scale]
    )
    
    scaler_dict[sym] = scaler

In [ ]:
# Step 2: Create sequences per symbol
def create_sequences(group, time_step=60):
    data = group[features + [target_col]].values
    X, y = [], []
    for i in range(len(data) - time_step):
        X.append(data[i:i + time_step, :-1])  # all features
        y.append(data[i + time_step, -1])     # close price
    return np.array(X), np.array(y)

X_all, y_all = [], []
for _, group in df.groupby("symbol"):
    X, y = create_sequences(group)
    X_all.append(X)
    y_all.append(y)

X = np.concatenate(X_all)
y = np.concatenate(y_all)

In [ ]:
y

In [ ]:
print("✅ Sequence data prepared:", X.shape, y.shape)

In [ ]:
# Step 3: Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, shuffle=False, test_size=0.2)

In [ ]:
# Step 4: Define LSTM Model
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(X.shape[1], X.shape[2])),
    Dropout(0.2),
    LSTM(64),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)
])

model.compile(optimizer='adam', loss='mean_squared_error')
model.summary()

In [ ]:
# Step 5: Train model
from tensorflow.keras.callbacks import EarlyStopping
early_stop = EarlyStopping(patience=10, restore_best_weights=True)
# Assuming you saved the history when training
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

# y_pred = model.predict(X_test)

In [ ]:
# Plot loss curve
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title("Training & Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Predict (already done)
y_pred = model.predict(X_test)

# Plot
plt.figure(figsize=(12, 5))
plt.plot(y_test, label='Actual')
plt.plot(y_pred, label='Predicted')
plt.title("Predicted vs Actual Close Prices (Normalized)")
plt.xlabel("Time Step")
plt.ylabel("Close (scaled)")
plt.legend()
plt.grid(True)
plt.show()